# tAilorCode — Colab development checks

Start with a CPU runtime. This notebook downloads the public project and installs its small validation dependency. It does not download Gemma weights, access Kaggle credentials, mount Drive, create cloud resources, or submit to the competition.

Run all cells. Packaging checks do not measure agent performance. If you later choose a GPU runtime, the final cell reports its hardware; GPU availability alone does not establish harness compatibility.


In [ ]:
from pathlib import Path
import subprocess
import sys

repository = Path('/content/tailorcode-development')
if repository.exists():
    raise RuntimeError('This checkout already exists. Use it as-is or choose a new directory; local changes will not be overwritten.')
subprocess.run(['git', 'clone', '--depth', '1',
                'https://github.com/AraceliFradejas/tailorcode-gemma-developer-agent.git',
                str(repository)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(repository / 'requirements-dev.txt')], check=True)
print('Checked-out commit:')
subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repository, check=True)


In [ ]:
for command in [
    [sys.executable, 'scripts/build_submission.py', '--check'],
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    [sys.executable, 'scripts/check_submission_notebook.py'],
]:
    subprocess.run(command, cwd=repository, check=True)
print('Development checks passed. No agent benchmark has been run.')


In [ ]:
import importlib.metadata
import json
import platform
import shutil

report = {'python': platform.python_version(), 'platform': platform.system(),
          'architecture': platform.machine(), 'gpu': None,
          'docker_command': shutil.which('docker'), 'packages': {},
          'agent_evaluation': 'not_run'}
for package in ['swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'vllm']:
    try:
        report['packages'][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        report['packages'][package] = None
if shutil.which('nvidia-smi'):
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                                 '--format=csv,noheader'], capture_output=True, text=True, timeout=15)
        report['gpu'] = result.stdout.strip() if result.returncode == 0 else 'GPU query failed'
    except subprocess.TimeoutExpired:
        report['gpu'] = 'GPU query timed out'
report_path = Path('/content/tailorcode-environment.json')
report_path.write_text(json.dumps(report, indent=2) + '\n')
print(json.dumps(report, indent=2))
print('Report saved:', report_path)
print('Do not interpret this report as confirmation that the full model fits or the harness can run.')


## Next step

Save or download `tailorcode-environment.json` before disconnecting: Colab runtime files are temporary. Use the report to decide whether this environment can support compiler validation or a limited inference experiment. The competition harness, sandbox, model availability, and memory requirements still need verification.

No paid plan is required for these development checks. Do not purchase compute solely to make this notebook pass.
